# Exercises 3 and 4 — TF-IDF and First Search Engine

Are all words equally informative? Learn weighting, shared query coordinates, cosine and ranking. Import shared implementations; do not duplicate algorithms. Reference cores 3.1–3.5 and 4.1–4.3 are complete.

In [ ]:
from pathlib import Path
import sys, csv
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.classical_nlp import build_vocabulary, build_document_term_matrix, calculate_tf, calculate_df, calculate_idf, calculate_tfidf, build_tfidf_matrix, build_query_tfidf_vector, cosine_similarity, sklearn_tfidf_comparison
from src.retrieval import tokenize_for_search, prepare_search_corpus, search_tfidf, score_documents, rank_documents

## 1–3. Documents → tokens → shared vocabulary
Preprocessing uses Exercise 1 through the shared search policy. Retain stopwords; no stemming or model resources. Predict which document should match natural language processing.

In [ ]:
texts = ["Natural language processing analyzes human language.", "Machine learning discovers patterns from data.", "Computer networks connect devices and exchange information.", "Database systems organize and query structured data."]
documents = [{"document_id": f"D{i+1}", "text": text} for i, text in enumerate(texts)]
tokens = [tokenize_for_search(text) for text in texts]
vocabulary = build_vocabulary(tokens)
tokens, vocabulary

## 4. TF — TODO 3.1
TF(t,d)=count(t,d)/document length. Empty documents yield zeros. Inspect counts before normalizing.

In [ ]:
bow = build_document_term_matrix(tokens, vocabulary)
tf = [calculate_tf(document, vocabulary) for document in tokens]
bow, tf

## 5. DF — TODO 3.2
Count documents containing a term, not repetitions within one document.

In [ ]:
df = calculate_df(tokens, vocabulary)
list(zip(vocabulary, df))

## 6. IDF — TODO 3.3
IDF=ln(N/DF), natural logarithm, unsmoothed. Corpus-wide terms receive zero. Zero DF is invalid: only corpus terms belong in this vocabulary.

In [ ]:
idf = calculate_idf(df, len(documents))
list(zip(vocabulary, df, idf))

## 7–8. TF-IDF — TODO 3.4
Multiply aligned TF and IDF; matrix assembly reuses the individual operation.

In [ ]:
matrix = build_tfidf_matrix(tokens, vocabulary, idf)
list(zip(texts, matrix))

## Paper-and-pencil check
D1=common nlp nlp, D2=common text, D3=common text. Before running, calculate D1 TF(nlp)=2/3, DF(nlp)=1, IDF(nlp)=ln(3), weight≈0.732408. common has weight zero everywhere.

In [ ]:
hand_tokens = [["common", "nlp", "nlp"], ["common", "text"], ["common", "text"]]
hand_vocabulary = build_vocabulary(hand_tokens)
hand_idf = calculate_idf(calculate_df(hand_tokens, hand_vocabulary), 3)
hand_vocabulary, hand_idf, build_tfidf_matrix(hand_tokens, hand_vocabulary, hand_idf)

## 9. Query — TODO 3.5
Use the corpus vocabulary and corpus IDF, never refit on the query. Ignore OOV terms and normalize over retained known tokens.

In [ ]:
query = "natural language processing"
query_tokens = tokenize_for_search(query)
query_vector = build_query_tfidf_vector(query_tokens, vocabulary, idf)
list(zip(vocabulary, query_vector))

## 10. Cosine — TODO 4.1
Dot product divided by the product of magnitudes. Direction matters more than vector length. A zero denominator returns zero similarity, not a probability.

In [ ]:
cosine_similarity([1, 1], [1, 0]), cosine_similarity(query_vector, matrix[0])

## 11–12. Score and rank — TODOs 4.2 and 4.3
Reuse cosine for each row, then sort descending. Equal scores retain original document order. Top K caps results; zero-score rows are displayed transparently, not treated as relevant evidence.

In [ ]:
scores = score_documents(query_vector, matrix)
rank_documents(documents, scores, top_k=3)

In [ ]:
corpus = prepare_search_corpus(documents)
search_tfidf(query, corpus, top_k=3)

## 13. Manual vs sklearn, after the manual calculations
Our TF=count/length and IDF=ln(N/DF). sklearn uses raw counts and smoothed IDF=ln((1+N)/(1+DF))+1, norm=None for inspection. Identical tokens and fixed columns do not imply equal values. Default sklearn norm is L2; smooth_idf=False still retains +1. Row normalization does not change cosine, but changed IDF may change direction/ranking.

In [ ]:
library = sklearn_tfidf_comparison(tokens, vocabulary)
list(zip(vocabulary, idf, library["idf"])), library["matrix"]

In [ ]:
library_query = build_query_tfidf_vector(query_tokens, vocabulary, library["idf"])
library_scores = score_documents(library_query, library["matrix"])
rank_documents(documents, library_scores, 3)

## 14. Search the course corpus
Index descriptions only, not titles or keywords; titles label results. This deliberate field choice preserves a realistic vocabulary mismatch. Do not change supplied descriptions to force success.

In [ ]:
with (ROOT / "data/search_corpus/course_descriptions.csv").open(encoding="utf-8", newline="") as stream:
    rows = list(csv.DictReader(stream))
course_documents = [{"document_id": row["course_code"], "title": row["course_name"], "text": row["description"]} for row in rows]
course_corpus = prepare_search_corpus(course_documents)
search_tfidf("natural language processing", course_corpus, 3)["results"]

## 15. Paraphrase and OOV experiments
Predict the winner; inspect which query words are absent. Try machine learning, database systems and computer communication networks too. High lexical overlap is not guaranteed semantic relevance.

In [ ]:
paraphrase = "Which subject teaches machines to work with human communication?"
search_tfidf(paraphrase, course_corpus, 3)

In [ ]:
oov_result = search_tfidf("zzworkshopunknown", course_corpus, 3)
assert not any(oov_result["query_vector"])
assert all(score == 0 for score in oov_result["scores"])
oov_result

## 16. Discuss limitations and checkpoint
Corpus-wide terms vanish; unseen words create no dimensions; morphological forms and synonyms remain separate. Retrieval ranks content but does not answer, summarize or analyze papers.

Checkpoint 3: You have now built a small information retrieval system using NLP and mathematics—without an LLM.

Run `python -m pytest`. Suggested commit: `git commit -m "Build TF-IDF search engine"`. Embeddings come later.